In [ ]:
!pip install -U bitsandbytes transformers accelerate qwen-vl-utils scikit-learn tqdm -q

In [22]:
!pip install zeus-ml -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 5.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 10.7 MB/s eta 0:00:00


In [1]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF token loaded:", os.environ["HF_TOKEN"][:8], "...")

HF token loaded: hf_GibuC ...


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Visible devices:", torch.cuda.device_count())
print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Visible devices: 1
Device: Tesla T4


In [5]:
# Cell A1 — load with EAGER attention (required to read attention weights)
import torch, os, json
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map={"": 0},
    attn_implementation="eager",        # <-- the key difference from Phase 2
)
model.eval()

# bound visual token count so attention tensors stay within 16GB
processor = AutoProcessor.from_pretrained(
    MODEL_ID, min_pixels=256*28*28, max_pixels=768*28*28
)
print("Loaded.")
print("Text-decoder layers:", model.config.text_config.num_hidden_layers)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded.
Text-decoder layers: 28


In [6]:
# Cell A2 — capture per-layer attention paid to visual tokens
import numpy as np
from PIL import Image

DATA_DIR = "/kaggle/input/notebooks/minhajurrahman223050/ecovlm-data-prep/vqav2_subset"
IMG_DIR = os.path.join(DATA_DIR, "images")
SUFFIX = "Answer the question using a single word or phrase."
N_PROBE = 200

with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    SAMPLES = json.load(f)[:N_PROBE]

IMG_TOKEN_ID = model.config.image_token_id

def token_attention_profile(sample):
    """Returns (n_layers, n_visual) array: attention each visual token
    receives from text query positions, averaged over heads."""
    image = Image.open(os.path.join(IMG_DIR, sample["image_file"])).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": f"{sample['question']} {SUFFIX}"}]}]
    text = processor.apply_chat_template(messages, tokenize=False,
                                         add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda:0")

    ids = inputs.input_ids[0]
    vis_mask = (ids == IMG_TOKEN_ID)
    vis_idx = torch.nonzero(vis_mask).squeeze(-1)
    if vis_idx.numel() == 0:
        return None, None
    vis_start, vis_end = vis_idx[0].item(), vis_idx[-1].item() + 1
    text_start = vis_end                      # text tokens follow the image block

    with torch.no_grad():
        out = model(**inputs, output_attentions=True, use_cache=False)

    profile = []
    for layer_attn in out.attentions:         # each: (1, heads, q, k)
        a = layer_attn[0].float()             # (heads, q, k)
        a = a.mean(dim=0)                     # avg over heads -> (q, k)
        a = a[text_start:, vis_start:vis_end]  # text queries -> visual keys
        profile.append(a.mean(dim=0).cpu().numpy())   # (n_visual,)

    del out
    torch.cuda.empty_cache()
    return np.stack(profile), vis_end - vis_start     # (n_layers, n_visual)

In [ ]:
# Cell A3 — run the probe
from tqdm.auto import tqdm

EARLY = slice(2, 9)      # layers 2-8
LATE  = slice(15, 26)    # layers 15-25

records = []
for s in tqdm(SAMPLES):
    try:
        prof, n_vis = token_attention_profile(s)
        if prof is None:
            continue
        early = prof[EARLY]                    # (7, n_vis)
        late  = prof[LATE]                     # (11, n_vis)
        records.append(dict(
            sample_idx=s["sample_idx"],
            n_visual=n_vis,
            early_mean=early.mean(axis=0),
            early_var=early.var(axis=0),
            early_cv=early.std(axis=0) / (early.mean(axis=0) + 1e-12),
            late_mean=late.mean(axis=0),
        ))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("OOM on", s["sample_idx"], "- lower max_pixels")
        continue

print("Collected:", len(records), "samples")
np.save("/kaggle/working/dap_probe.npy", np.array(records, dtype=object),
        allow_pickle=True)

In [ ]:
# Cell A4 — the actual test
from sklearn.metrics import roc_auc_score
import numpy as np

LOW_MEAN_PCTILE = 50    # "looks unimportant early" = bottom half
IMPORTANT_PCTILE = 75   # "became important late" = top quartile by late attention

aucs_mean, aucs_var, aucs_both = [], [], []

for r in records:
    em, ev, lm = r["early_mean"], r["early_var"], r["late_mean"]

    # restrict to the pruning-candidate pool: low early attention
    pool = em <= np.percentile(em, LOW_MEAN_PCTILE)
    if pool.sum() < 20:
        continue

    label = (lm[pool] >= np.percentile(lm, IMPORTANT_PCTILE)).astype(int)
    if label.sum() == 0 or label.sum() == len(label):
        continue    # degenerate, no signal to measure

    # predictor 1: early mean alone (what every existing method uses)
    a_mean = roc_auc_score(label, em[pool])
    # predictor 2: variance alone
    a_var  = roc_auc_score(label, ev[pool])
    # predictor 3: both, simple standardised sum
    z = lambda x: (x - x.mean()) / (x.std() + 1e-12)
    a_both = roc_auc_score(label, z(em[pool]) + z(ev[pool]))

    aucs_mean.append(a_mean); aucs_var.append(a_var); aucs_both.append(a_both)

import statistics as st
print(f"Samples analysed: {len(aucs_mean)}")
print(f"AUC, early mean alone : {st.mean(aucs_mean):.4f}  (baseline)")
print(f"AUC, variance alone   : {st.mean(aucs_var):.4f}")
print(f"AUC, mean + variance  : {st.mean(aucs_both):.4f}")
delta = st.mean(aucs_both) - st.mean(aucs_mean)
print(f"\nDELTA (the number that decides): {delta:+.4f}")
print("VERDICT:", "STRONG PASS" if delta >= 0.05
      else "WEAK PASS" if delta >= 0.02 else "FAIL — pivot needed")

In [ ]:
# Cell A5 — test better combination strategies on existing data, no GPU work
import numpy as np
from sklearn.metrics import roc_auc_score

aucs_var_only, aucs_weighted, aucs_gated = [], [], []

for r in records:
    em, ev, lm = r["early_mean"], r["early_var"], r["late_mean"]
    pool = em <= np.percentile(em, 50)
    if pool.sum() < 20: continue
    label = (lm[pool] >= np.percentile(lm, 75)).astype(int)
    if label.sum() == 0 or label.sum() == len(label): continue

    z = lambda x: (x - x.mean()) / (x.std() + 1e-12)

    # variance alone (confirmed strongest single predictor)
    aucs_var_only.append(roc_auc_score(label, ev[pool]))

    # weighted toward variance (80/20 instead of 50/50)
    aucs_weighted.append(roc_auc_score(label, 0.2*z(em[pool]) + 0.8*z(ev[pool])))

    # variance gated by a mean floor: only count variance where mean isn't near-zero
    floor = em[pool] > np.percentile(em[pool], 10)
    gated_score = ev[pool] * floor
    aucs_gated.append(roc_auc_score(label, gated_score))

import statistics as st
print(f"Variance alone:        {st.mean(aucs_var_only):.4f}")
print(f"80/20 weighted (var):  {st.mean(aucs_weighted):.4f}")
print(f"Variance, mean-gated:  {st.mean(aucs_gated):.4f}")

In [ ]:
# Cell A6 — minimal token-removal coherence test
import torch
from PIL import Image
import os, json

DATA_DIR = "/kaggle/input/notebooks/minhajurrahman223050/ecovlm-data-prep/vqav2_subset"
IMG_DIR = os.path.join(DATA_DIR, "images")

# Find the actual text-decoder submodule — verify the attribute path
print("model.model type:", type(model.model))
print("Has language_model attr:", hasattr(model.model, "language_model"))
if hasattr(model.model, "language_model"):
    text_model = model.model.language_model
else:
    text_model = model.model
print("text_model type:", type(text_model))
print("Number of layers:", len(text_model.layers))

In [ ]:
# Cell A7 — minimal fixed-index token removal coherence test
import torch
from PIL import Image
import os, json

DATA_DIR = "/kaggle/input/notebooks/minhajurrahman223050/ecovlm-data-prep/vqav2_subset"
IMG_DIR = os.path.join(DATA_DIR, "images")

with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    SAMPLES = json.load(f)

text_model = model.model.language_model
IMG_TOKEN_ID = model.config.image_token_id

def run_with_dropped_tokens(sample, drop_frac=0.0, drop_layer=8):
    """drop_frac=0.0 -> baseline, no tokens removed.
    drop_frac>0 -> removes that fraction of visual tokens (fixed, arbitrary
    indices - no scoring logic yet) right before drop_layer runs."""
    image = Image.open(os.path.join(IMG_DIR, sample["image_file"])).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": f"{sample['question']} Answer the question using a single word or phrase."}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda:0")

    ids = inputs.input_ids[0]
    vis_mask = (ids == IMG_TOKEN_ID)
    vis_idx = torch.nonzero(vis_mask).squeeze(-1)
    vis_start, vis_end = vis_idx[0].item(), vis_idx[-1].item() + 1
    n_vis = vis_end - vis_start

    if drop_frac == 0.0:
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
        gen = out[:, inputs.input_ids.shape[1]:]
        return processor.batch_decode(gen, skip_special_tokens=True)[0].strip(), n_vis, 0

    # fixed, arbitrary "drop" set: every other visual token in the back half
    # (deliberately naive - real scoring comes later)
    candidate = torch.arange(vis_start + n_vis // 2, vis_end, 2)
    n_drop = int(len(candidate) * drop_frac / 0.5)  # scale roughly to drop_frac
    drop_idx = set(candidate[:n_drop].tolist())

    keep_mask = torch.ones(ids.shape[0], dtype=torch.bool)
    for idx in drop_idx:
        keep_mask[idx] = False
    keep_positions = torch.nonzero(keep_mask).squeeze(-1)

    print(f"  Total tokens: {ids.shape[0]}, visual: {n_vis}, dropping: {len(drop_idx)}")

    # Manually replicate the embed + position + generate steps with pruned inputs
    with torch.no_grad():
        pruned_ids = inputs.input_ids[:, keep_positions]
        pruned_pixel_values = inputs.get("pixel_values")
        pruned_grid_thw = inputs.get("image_grid_thw")

        gen_kwargs = dict(max_new_tokens=32, do_sample=False)
        out = model.generate(
            input_ids=pruned_ids,
            attention_mask=torch.ones_like(pruned_ids),
            pixel_values=pruned_pixel_values,
            image_grid_thw=pruned_grid_thw,
            **gen_kwargs,
        )
    gen = out[:, pruned_ids.shape[1]:]
    return processor.batch_decode(gen, skip_special_tokens=True)[0].strip(), n_vis, len(drop_idx)


# Run on 5 samples: baseline vs. crude 50%-of-back-half dropped
print("="*60)
for s in SAMPLES[:5]:
    baseline, n_vis, _ = run_with_dropped_tokens(s, drop_frac=0.0)
    pruned, _, n_dropped = run_with_dropped_tokens(s, drop_frac=0.5)
    match = "SAME" if baseline.lower() == pruned.lower() else "DIFFERENT"
    print(f"Q: {s['question']}")
    print(f"  baseline: {baseline!r}")
    print(f"  pruned  : {pruned!r}   ({n_dropped} tokens dropped)  [{match}]")
    print("-"*60)

In [ ]:
# Cell A8 — corrected test: prune inputs_embeds AFTER vision encoding succeeds
import torch

def run_with_embeds_pruned(sample, drop_frac=0.0):
    image = Image.open(os.path.join(IMG_DIR, sample["image_file"])).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": f"{sample['question']} Answer the question using a single word or phrase."}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda:0")

    ids = inputs.input_ids[0]
    vis_mask = (ids == IMG_TOKEN_ID)
    vis_idx = torch.nonzero(vis_mask).squeeze(-1)
    vis_start, vis_end = vis_idx[0].item(), vis_idx[-1].item() + 1
    n_vis = vis_end - vis_start

    with torch.no_grad():
        # let the model build embeddings the normal way first (vision tower + placeholder fill)
        inputs_embeds = model.get_input_embeddings()(inputs.input_ids)
        image_embeds = model.get_image_features(inputs.pixel_values, inputs.image_grid_thw).pooler_output
        image_embeds = torch.cat(image_embeds, dim=0).to(inputs_embeds.device, inputs_embeds.dtype)
        mask, _ = model.model.get_placeholder_mask(inputs.input_ids, inputs_embeds=inputs_embeds, image_features=image_embeds)
        inputs_embeds = inputs_embeds.masked_scatter(mask, image_embeds)

        n_drop = int(n_vis * drop_frac)
        if n_drop > 0:
            candidate = torch.arange(vis_start + n_vis // 2, vis_end, 2)[:n_drop]
            keep_mask = torch.ones(inputs_embeds.shape[1], dtype=torch.bool)
            keep_mask[candidate] = False
            inputs_embeds = inputs_embeds[:, keep_mask, :]
            print(f"  Pruned {n_drop} tokens -> new seq len {inputs_embeds.shape[1]} (was {ids.shape[0]})")

        out = model.generate(inputs_embeds=inputs_embeds, max_new_tokens=32, do_sample=False)

    return processor.batch_decode(out, skip_special_tokens=True)[0].strip(), n_vis, n_drop


print("="*60)
for s in SAMPLES[:5]:
    baseline, n_vis, _ = run_with_embeds_pruned(s, drop_frac=0.0)
    pruned, _, n_dropped = run_with_embeds_pruned(s, drop_frac=0.25)
    match = "SAME" if baseline.lower() == pruned.lower() else "DIFFERENT"
    print(f"Q: {s['question']}")
    print(f"  baseline: {baseline!r}")
    print(f"  pruned  : {pruned!r}   ({n_dropped}/{n_vis} visual tokens dropped)  [{match}]")
    print("-"*60)

In [ ]:
# Cell A9 — stress test: drop fraction x drop strategy, on 25 samples, real VQA accuracy
import torch, re

def vqa_accuracy(predicted, ground_truth_answers):
    def normalise(s):
        s = s.lower().strip()
        s = re.sub(r"[^\w\s]", "", s)
        for a in ["a ", "an ", "the "]:
            if s.startswith(a): s = s[len(a):]
        return s.strip()
    p = normalise(predicted)
    m = sum(1 for a in ground_truth_answers if normalise(a) == p)
    return min(m / 3, 1.0)


def get_index_set(strategy, vis_start, vis_end, n_vis, drop_frac):
    n_drop = int(n_vis * drop_frac)
    if n_drop == 0:
        return torch.tensor([], dtype=torch.long)
    if strategy == "back_half_stride":       # our earlier placeholder rule
        cand = torch.arange(vis_start + n_vis // 2, vis_end, 2)
        return cand[:n_drop]
    if strategy == "first_chunk":            # deliberately bad: drop the start
        return torch.arange(vis_start, vis_start + n_drop)
    if strategy == "random":                 # unbiased baseline
        perm = torch.randperm(n_vis) + vis_start
        return perm[:n_drop]
    raise ValueError(strategy)


def run_pruned(sample, strategy, drop_frac):
    image = Image.open(os.path.join(IMG_DIR, sample["image_file"])).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": f"{sample['question']} Answer the question using a single word or phrase."}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda:0")

    ids = inputs.input_ids[0]
    vis_mask = (ids == IMG_TOKEN_ID)
    vis_idx = torch.nonzero(vis_mask).squeeze(-1)
    vis_start, vis_end = vis_idx[0].item(), vis_idx[-1].item() + 1
    n_vis = vis_end - vis_start

    with torch.no_grad():
        inputs_embeds = model.get_input_embeddings()(inputs.input_ids)
        image_embeds = model.get_image_features(inputs.pixel_values, inputs.image_grid_thw).pooler_output
        image_embeds = torch.cat(image_embeds, dim=0).to(inputs_embeds.device, inputs_embeds.dtype)
        mask, _ = model.model.get_placeholder_mask(inputs.input_ids, inputs_embeds=inputs_embeds, image_features=image_embeds)
        inputs_embeds = inputs_embeds.masked_scatter(mask, image_embeds)

        if drop_frac > 0:
            drop_idx = get_index_set(strategy, vis_start, vis_end, n_vis, drop_frac)
            keep_mask = torch.ones(inputs_embeds.shape[1], dtype=torch.bool)
            keep_mask[drop_idx.long()] = False
            inputs_embeds = inputs_embeds[:, keep_mask, :]

        out = model.generate(inputs_embeds=inputs_embeds, max_new_tokens=32, do_sample=False)

    return processor.batch_decode(out, skip_special_tokens=True)[0].strip()


N_STRESS = 25
DROP_FRACS = [0.0, 0.25, 0.50, 0.75]
STRATEGIES = ["random", "back_half_stride", "first_chunk"]

stress_samples = SAMPLES[:N_STRESS]
results = {}

for strat in STRATEGIES:
    for frac in DROP_FRACS:
        key = (strat, frac)
        if frac == 0.0 and "baseline" in [k[0] for k in results]:
            results[key] = results[("baseline", 0.0)]
            continue
        accs = []
        for s in stress_samples:
            pred = run_pruned(s, strat if frac > 0 else "random", frac)
            accs.append(vqa_accuracy(pred, s["answers"]))
        results[key] = sum(accs) / len(accs)
        print(f"{strat:18s} drop={frac:.2f}  acc={results[key]:.3f}")
    print()

print("="*60)
print("SUMMARY TABLE")
print("="*60)
print(f"{'drop_frac':>10s} | {'random':>8s} | {'back_half':>10s} | {'first_chunk':>12s}")
for frac in DROP_FRACS:
    r = results.get(("random", frac), results.get(("baseline", 0.0)))
    b = results.get(("back_half_stride", frac), r)
    f = results.get(("first_chunk", frac), r)
    print(f"{frac:>10.2f} | {r:>8.3f} | {b:>10.3f} | {f:>12.3f}")

In [ ]:
print("model:", "model" in dir())
print("processor:", "processor" in dir())
print("SAMPLES:", "SAMPLES" in dir())
print("IMG_DIR:", "IMG_DIR" in dir())

In [13]:
# Cell D1 — DAP forward-pass scaffold, 0% pruning (fidelity test)
import torch
from transformers.models.qwen2_5_vl.modeling_qwen2_5_vl import Qwen2_5_VLTextModel
from transformers.modeling_outputs import BaseModelOutputWithPast

CHECKPOINT_LAYERS = [8, 11, 14]   # Qwen's 28-layer schedule

def apply_checkpoint_stub(hidden_states, position_embeddings, causal_mask_mapping, survivor_idx):
    """
    Slices hidden_states, position_embeddings (cos, sin), and every mask in
    causal_mask_mapping down to survivor_idx. With survivor_idx = full range,
    this is a pure identity operation - output must be unchanged.
    """
    hidden_states = hidden_states[:, survivor_idx, :]

    cos, sin = position_embeddings
    cos = cos[..., survivor_idx, :]
    sin = sin[..., survivor_idx, :]
    position_embeddings = (cos, sin)

    new_mask_mapping = {}
    for key, mask in causal_mask_mapping.items():
        # mask shape: (batch, 1, q_len, k_len) - slice both query and key dims
        new_mask_mapping[key] = mask[:, :, survivor_idx, :][:, :, :, survivor_idx]
    causal_mask_mapping = new_mask_mapping

    return hidden_states, position_embeddings, causal_mask_mapping


def dap_forward(
    self,
    input_ids=None,
    attention_mask=None,
    position_ids=None,
    past_key_values=None,
    inputs_embeds=None,
    use_cache=None,
    **kwargs,
):
    if inputs_embeds is None:
        inputs_embeds = self.embed_tokens(input_ids)

    if position_ids is None:
        past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
        position_ids = torch.arange(inputs_embeds.shape[1], device=inputs_embeds.device) + past_seen_tokens
        position_ids = position_ids.view(1, 1, -1).expand(3, inputs_embeds.shape[0], -1)
    elif position_ids.ndim == 2:
        position_ids = position_ids[None, ...].expand(3, position_ids.shape[0], -1)

    text_position_ids = None
    if position_ids.ndim == 3 and position_ids.shape[0] == 4:
        text_position_ids = position_ids[0]
        position_ids = position_ids[1:]

    if not isinstance(causal_mask_mapping := attention_mask, dict):
        from transformers.masking_utils import create_causal_mask, create_sliding_window_causal_mask
        mask_kwargs = {
            "config": self.config, "inputs_embeds": inputs_embeds,
            "attention_mask": attention_mask, "past_key_values": past_key_values,
            "position_ids": text_position_ids,
        }
        causal_mask_mapping = {"full_attention": create_causal_mask(**mask_kwargs)}
        if self.has_sliding_layers:
            causal_mask_mapping["sliding_attention"] = create_sliding_window_causal_mask(**mask_kwargs)

    hidden_states = inputs_embeds
    position_embeddings = self.rotary_emb(hidden_states, position_ids)

    # survivor set starts as everyone; DAP would shrink this at each checkpoint
    seq_len = hidden_states.shape[1]
    survivor_idx = torch.arange(seq_len, device=hidden_states.device)

    for i, decoder_layer in enumerate(self.layers):
        hidden_states = decoder_layer(
            hidden_states,
            attention_mask=causal_mask_mapping[self.config.layer_types[i]],
            position_embeddings=position_embeddings,
            position_ids=text_position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            **kwargs,
        )

        if i in CHECKPOINT_LAYERS:
            # --- DAP would compute scores and shrink survivor_idx here ---
            # For this fidelity test: survivor_idx stays the full range (0% pruning)
            hidden_states, position_embeddings, causal_mask_mapping = apply_checkpoint_stub(
                hidden_states, position_embeddings, causal_mask_mapping, survivor_idx
            )
            print(f"  [checkpoint @ layer {i}] survivors: {len(survivor_idx)} / {seq_len}")

    hidden_states = self.norm(hidden_states)
    return BaseModelOutputWithPast(last_hidden_state=hidden_states, past_key_values=past_key_values)

In [14]:
# Cell D2 — fidelity test: baseline vs. DAP-scaffolded forward, must match exactly
import types

text_model = model.model.language_model

s = SAMPLES[0]
image = Image.open(os.path.join(IMG_DIR, s["image_file"])).convert("RGB")
messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": f"{s['question']} Answer the question using a single word or phrase."}]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda:0")

with torch.no_grad():
    inputs_embeds = model.get_input_embeddings()(inputs.input_ids)
    image_embeds = model.get_image_features(inputs.pixel_values, inputs.image_grid_thw).pooler_output
    image_embeds = torch.cat(image_embeds, dim=0).to(inputs_embeds.device, inputs_embeds.dtype)
    mask, _ = model.model.get_placeholder_mask(inputs.input_ids, inputs_embeds=inputs_embeds, image_features=image_embeds)
    inputs_embeds = inputs_embeds.masked_scatter(mask, image_embeds)

    print("--- BASELINE forward ---")
    out_baseline = text_model(inputs_embeds=inputs_embeds, use_cache=False)
    logits_baseline = model.lm_head(out_baseline.last_hidden_state[:, -1, :])

    print("--- DAP-scaffolded forward (0% pruning) ---")
    original_forward = text_model.forward
    text_model.forward = types.MethodType(dap_forward, text_model)
    out_dap = text_model(inputs_embeds=inputs_embeds, use_cache=False)
    text_model.forward = original_forward
    logits_dap = model.lm_head(out_dap.last_hidden_state[:, -1, :])

max_diff = (logits_baseline - logits_dap).abs().max().item()
match = logits_baseline.argmax(-1).item() == logits_dap.argmax(-1).item()
print(f"\nNext-token match: {match}")
print(f"Max logit difference: {max_diff}")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:944: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


--- BASELINE forward ---
--- DAP-scaffolded forward (0% pruning) ---
  [checkpoint @ layer 8] survivors: 304 / 304
  [checkpoint @ layer 11] survivors: 304 / 304
  [checkpoint @ layer 14] survivors: 304 / 304

Next-token match: True
Max logit difference: 0.0


In [15]:
# Cell D3 — real pruning at checkpoint 1 only (checkpoints 2, 3 still no-ops, FIXED)
import torch, types
import numpy as np
from transformers.modeling_outputs import BaseModelOutputWithPast

EARLY_WINDOW = list(range(2, 9))   # layers 2-8, feeds checkpoint 1's score
CHECKPOINT_LAYERS = [8, 11, 14]
DROP_FRAC = 0.5                     # single fixed cutoff for this test - not yet tuned

attn_capture = {}

def make_capture_hook(layer_idx):
    def hook(module, inputs, output):
        if isinstance(output, tuple) and len(output) > 1 and output[1] is not None:
            attn_capture[layer_idx] = output[1][0].float().mean(dim=0).detach().cpu().numpy()  # (q,k)
        return output
    return hook

def register_early_hooks(text_model, layers):
    handles = []
    for i in layers:
        h = text_model.layers[i].self_attn.register_forward_hook(make_capture_hook(i))
        handles.append(h)
    return handles

def unregister_hooks(handles):
    for h in handles:
        h.remove()


def apply_checkpoint_stub(hidden_states, position_embeddings, causal_mask_mapping, survivor_idx):
    hidden_states = hidden_states[:, survivor_idx, :]
    cos, sin = position_embeddings
    cos = cos[..., survivor_idx, :]
    sin = sin[..., survivor_idx, :]
    position_embeddings = (cos, sin)
    new_mask_mapping = {}
    for key, mask in causal_mask_mapping.items():
        new_mask_mapping[key] = mask[:, :, survivor_idx, :][:, :, :, survivor_idx]
    causal_mask_mapping = new_mask_mapping
    return hidden_states, position_embeddings, causal_mask_mapping


def dap_forward_v2(
    self, input_ids=None, attention_mask=None, position_ids=None,
    past_key_values=None, inputs_embeds=None, use_cache=None, **kwargs,
):
    if inputs_embeds is None:
        inputs_embeds = self.embed_tokens(input_ids)

    if position_ids is None:
        past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
        position_ids = torch.arange(inputs_embeds.shape[1], device=inputs_embeds.device) + past_seen_tokens
        position_ids = position_ids.view(1, 1, -1).expand(3, inputs_embeds.shape[0], -1)
    elif position_ids.ndim == 2:
        position_ids = position_ids[None, ...].expand(3, position_ids.shape[0], -1)

    text_position_ids = None
    if position_ids.ndim == 3 and position_ids.shape[0] == 4:
        text_position_ids = position_ids[0]
        position_ids = position_ids[1:]

    if not isinstance(causal_mask_mapping := attention_mask, dict):
        from transformers.masking_utils import create_causal_mask, create_sliding_window_causal_mask
        mask_kwargs = {
            "config": self.config, "inputs_embeds": inputs_embeds,
            "attention_mask": attention_mask, "past_key_values": past_key_values,
            "position_ids": text_position_ids,
        }
        causal_mask_mapping = {"full_attention": create_causal_mask(**mask_kwargs)}
        if self.has_sliding_layers:
            causal_mask_mapping["sliding_attention"] = create_sliding_window_causal_mask(**mask_kwargs)

    hidden_states = inputs_embeds
    position_embeddings = self.rotary_emb(hidden_states, position_ids)

    seq_len = hidden_states.shape[1]
    survivor_idx = torch.arange(seq_len, device=hidden_states.device)

    vis_start, vis_end, text_start = self._dap_vis_range

    for i, decoder_layer in enumerate(self.layers):
        hidden_states = decoder_layer(
            hidden_states,
            attention_mask=causal_mask_mapping[self.config.layer_types[i]],
            position_embeddings=position_embeddings,
            position_ids=text_position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            output_attentions=(i in EARLY_WINDOW),
            **kwargs,
        )

        if i == CHECKPOINT_LAYERS[0]:  # layer 8 - real scoring happens here
            vis_scores = []
            for layer_idx in EARLY_WINDOW:
                if layer_idx in attn_capture:
                    a = attn_capture[layer_idx][text_start:, vis_start:vis_end]
                    vis_scores.append(a.mean(axis=0))
            stacked = np.stack(vis_scores)
            variance_score = stacked.var(axis=0)

            n_vis = vis_end - vis_start
            n_drop = int(n_vis * DROP_FRAC)
            drop_rank = np.argsort(variance_score)
            drop_local_idx = drop_rank[:n_drop]
            drop_global_idx = torch.tensor(drop_local_idx + vis_start, device=hidden_states.device)

            keep_mask = torch.ones(seq_len, dtype=torch.bool, device=hidden_states.device)
            keep_mask[drop_global_idx.long()] = False
            survivor_idx = torch.nonzero(keep_mask).squeeze(-1)

            print(f"  [checkpoint @ layer {i}] dropped {n_drop}/{n_vis} visual tokens "
                  f"-> survivors: {len(survivor_idx)}/{seq_len}")

            hidden_states, position_embeddings, causal_mask_mapping = apply_checkpoint_stub(
                hidden_states, position_embeddings, causal_mask_mapping, survivor_idx
            )

        elif i in CHECKPOINT_LAYERS[1:]:  # layers 11, 14 - no-ops for this test
            # FIX: survivor_idx must be freshly full-range for the CURRENT (already-shrunk)
            # sequence length, not the stale original-space tensor from checkpoint 1.
            current_len = hidden_states.shape[1]
            survivor_idx = torch.arange(current_len, device=hidden_states.device)
            hidden_states, position_embeddings, causal_mask_mapping = apply_checkpoint_stub(
                hidden_states, position_embeddings, causal_mask_mapping, survivor_idx
            )
            print(f"  [checkpoint @ layer {i}] no-op, survivors: {len(survivor_idx)}/{current_len}")

    hidden_states = self.norm(hidden_states)
    return BaseModelOutputWithPast(last_hidden_state=hidden_states, past_key_values=past_key_values)

In [ ]:
# Cell D4 — test: real 50% pruning at checkpoint 1, vs baseline, on 5 samples
IMG_TOKEN_ID = model.config.image_token_id
text_model = model.model.language_model

def run_dap_test(sample, use_dap):
    image = Image.open(os.path.join(IMG_DIR, sample["image_file"])).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": f"{sample['question']} Answer the question using a single word or phrase."}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda:0")

    ids = inputs.input_ids[0]
    vis_mask = (ids == IMG_TOKEN_ID)
    vis_idx = torch.nonzero(vis_mask).squeeze(-1)
    vis_start, vis_end = vis_idx[0].item(), vis_idx[-1].item() + 1
    text_start = vis_end

    with torch.no_grad():
        inputs_embeds = model.get_input_embeddings()(inputs.input_ids)
        image_embeds = model.get_image_features(inputs.pixel_values, inputs.image_grid_thw).pooler_output
        image_embeds = torch.cat(image_embeds, dim=0).to(inputs_embeds.device, inputs_embeds.dtype)
        mask, _ = model.model.get_placeholder_mask(inputs.input_ids, inputs_embeds=inputs_embeds, image_features=image_embeds)
        inputs_embeds = inputs_embeds.masked_scatter(mask, image_embeds)

        hooks = []
        if use_dap:
            attn_capture.clear()
            hooks = register_early_hooks(text_model, EARLY_WINDOW)
            text_model._dap_vis_range = (vis_start, vis_end, text_start)
            original_forward = text_model.forward
            text_model.forward = types.MethodType(dap_forward_v2, text_model)

        current_embeds = inputs_embeds
        generated = None
        for step in range(31):
            out = text_model(inputs_embeds=current_embeds, use_cache=False)
            next_logits = model.lm_head(out.last_hidden_state[:, -1, :])
            next_tok = next_logits.argmax(-1)
            if next_tok.item() == processor.tokenizer.eos_token_id:
                break
            next_embed = model.get_input_embeddings()(next_tok.unsqueeze(0))
            current_embeds = torch.cat([current_embeds, next_embed], dim=1)
            generated = next_tok if generated is None else torch.cat([generated, next_tok])

        if use_dap:
            text_model.forward = original_forward
            unregister_hooks(hooks)

    if generated is None:
        return ""
    return processor.tokenizer.decode(generated, skip_special_tokens=True).strip()


print("="*60)
for s in SAMPLES[:5]:
    baseline = run_dap_test(s, use_dap=False)
    pruned = run_dap_test(s, use_dap=True)
    match = "SAME" if baseline.lower() == pruned.lower() else "DIFFERENT"
    print(f"Q: {s['question']}")
    print(f"  baseline: {baseline!r}")
    print(f"  pruned  : {pruned!r}  [{match}]")
    print("-"*60)

In [ ]:
# Cell D5 — larger accuracy check: baseline vs. DAP (checkpoint 1, 50% cutoff), N=40
import re

def vqa_accuracy(predicted, ground_truth_answers):
    def normalise(s):
        s = s.lower().strip()
        s = re.sub(r"[^\w\s]", "", s)
        for a in ["a ", "an ", "the "]:
            if s.startswith(a): s = s[len(a):]
        return s.strip()
    p = normalise(predicted)
    m = sum(1 for a in ground_truth_answers if normalise(a) == p)
    return min(m / 3, 1.0)


N_CHECK = 40
results = []

for idx, s in enumerate(SAMPLES[:N_CHECK]):
    baseline_pred = run_dap_test(s, use_dap=False)
    pruned_pred = run_dap_test(s, use_dap=True)

    baseline_acc = vqa_accuracy(baseline_pred, s["answers"])
    pruned_acc = vqa_accuracy(pruned_pred, s["answers"])

    results.append({
        "idx": idx,
        "question": s["question"],
        "baseline_pred": baseline_pred,
        "pruned_pred": pruned_pred,
        "baseline_acc": baseline_acc,
        "pruned_acc": pruned_acc,
        "match": baseline_pred.lower() == pruned_pred.lower(),
    })

    if (idx + 1) % 5 == 0:
        print(f"  ...{idx+1}/{N_CHECK} done")

print("\n" + "="*60)
print("SUMMARY")
print("="*60)

import statistics as st
baseline_mean = st.mean(r["baseline_acc"] for r in results)
pruned_mean = st.mean(r["pruned_acc"] for r in results)
exact_match_rate = sum(r["match"] for r in results) / len(results)

print(f"N = {len(results)}")
print(f"Baseline accuracy: {baseline_mean:.4f}")
print(f"Pruned accuracy:   {pruned_mean:.4f}")
print(f"Accuracy delta:    {pruned_mean - baseline_mean:+.4f}")
print(f"Exact prediction match rate: {exact_match_rate:.2%}")

print("\n--- Cases where predictions DIFFERED ---")
diffs = [r for r in results if not r["match"]]
if diffs:
    for r in diffs:
        print(f"  Q: {r['question']}")
        print(f"    baseline: {r['baseline_pred']!r} (acc={r['baseline_acc']:.2f})")
        print(f"    pruned  : {r['pruned_pred']!r} (acc={r['pruned_acc']:.2f})")
else:
    print("  None - all predictions matched exactly.")

In [17]:
# Cell D6 — checkpoints 1 & 2 with REAL deferral logic (checkpoint 3 still forced-keep, next step)
import torch, types
import numpy as np
from transformers.modeling_outputs import BaseModelOutputWithPast

CHECKPOINT_LAYERS = [8, 11, 14]
KEEP_PCTILE = 70    # mean percentile above which a token is kept outright
DROP_PCTILE = 40    # variance percentile (within low-mean pool) below which a token drops

attn_capture = {}

def make_capture_hook(layer_idx):
    def hook(module, inputs, output):
        if isinstance(output, tuple) and len(output) > 1 and output[1] is not None:
            attn_capture[layer_idx] = output[1][0].float().mean(dim=0).detach().cpu().numpy()
        return output
    return hook

def register_hooks(text_model, layers):
    return [text_model.layers[i].self_attn.register_forward_hook(make_capture_hook(i)) for i in layers]

def unregister_hooks(handles):
    for h in handles:
        h.remove()

def apply_checkpoint_stub(hidden_states, position_embeddings, causal_mask_mapping, survivor_idx):
    hidden_states = hidden_states[:, survivor_idx, :]
    cos, sin = position_embeddings
    cos, sin = cos[..., survivor_idx, :], sin[..., survivor_idx, :]
    new_mask = {k: m[:, :, survivor_idx, :][:, :, :, survivor_idx] for k, m in causal_mask_mapping.items()}
    return hidden_states, (cos, sin), new_mask


def score_and_gate(candidate_local_idx, mean_vals, var_vals, force_resolve=False):
    """
    candidate_local_idx: indices (into current sequence) of tokens still eligible for scoring
    mean_vals, var_vals:  arrays aligned with candidate_local_idx
    Returns: keep_idx, drop_idx, defer_idx (all as candidate_local_idx subsets)
    """
    n = len(candidate_local_idx)
    if n == 0:
        return np.array([], dtype=int), np.array([], dtype=int), np.array([], dtype=int)

    mean_cut = np.percentile(mean_vals, KEEP_PCTILE)
    high_mean_mask = mean_vals >= mean_cut
    keep_idx = candidate_local_idx[high_mean_mask]

    low_mean_local = candidate_local_idx[~high_mean_mask]
    low_mean_var = var_vals[~high_mean_mask]

    if len(low_mean_local) == 0:
        return keep_idx, np.array([], dtype=int), np.array([], dtype=int)

    if force_resolve:
        # no defer allowed: split the remaining pool at its median variance
        var_cut = np.median(low_mean_var)
        drop_mask = low_mean_var <= var_cut
        drop_idx = low_mean_local[drop_mask]
        keep_idx = np.concatenate([keep_idx, low_mean_local[~drop_mask]])
        defer_idx = np.array([], dtype=int)
    else:
        var_cut = np.percentile(low_mean_var, DROP_PCTILE)
        drop_mask = low_mean_var <= var_cut
        drop_idx = low_mean_local[drop_mask]
        defer_idx = low_mean_local[~drop_mask]

    return keep_idx, drop_idx, defer_idx


def dap_forward_v3(self, input_ids=None, attention_mask=None, position_ids=None,
                    past_key_values=None, inputs_embeds=None, use_cache=None, **kwargs):
    if inputs_embeds is None:
        inputs_embeds = self.embed_tokens(input_ids)
    if position_ids is None:
        past_seen = past_key_values.get_seq_length() if past_key_values is not None else 0
        position_ids = torch.arange(inputs_embeds.shape[1], device=inputs_embeds.device) + past_seen
        position_ids = position_ids.view(1, 1, -1).expand(3, inputs_embeds.shape[0], -1)
    elif position_ids.ndim == 2:
        position_ids = position_ids[None, ...].expand(3, position_ids.shape[0], -1)

    text_position_ids = None
    if position_ids.ndim == 3 and position_ids.shape[0] == 4:
        text_position_ids = position_ids[0]
        position_ids = position_ids[1:]

    if not isinstance(causal_mask_mapping := attention_mask, dict):
        from transformers.masking_utils import create_causal_mask, create_sliding_window_causal_mask
        mask_kwargs = {"config": self.config, "inputs_embeds": inputs_embeds,
                       "attention_mask": attention_mask, "past_key_values": past_key_values,
                       "position_ids": text_position_ids}
        causal_mask_mapping = {"full_attention": create_causal_mask(**mask_kwargs)}
        if self.has_sliding_layers:
            causal_mask_mapping["sliding_attention"] = create_sliding_window_causal_mask(**mask_kwargs)

    hidden_states = inputs_embeds
    position_embeddings = self.rotary_emb(hidden_states, position_ids)

    vis_start, vis_end, text_start = self._dap_vis_range
    # deferred_local: indices (into the CURRENT, possibly-shrunk sequence) of visual
    # tokens still eligible for scoring. Starts as the full visual range.
    deferred_local = np.arange(vis_start, vis_end)
    layer_attn_history = {}   # layer_idx -> (q,k) captured, reset per checkpoint window

    for i, decoder_layer in enumerate(self.layers):
        hidden_states = decoder_layer(
            hidden_states,
            attention_mask=causal_mask_mapping[self.config.layer_types[i]],
            position_embeddings=position_embeddings,
            position_ids=text_position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            output_attentions=True,
            **kwargs,
        )

        if i in CHECKPOINT_LAYERS:
            ckpt_idx = CHECKPOINT_LAYERS.index(i)
            is_final = (ckpt_idx == len(CHECKPOINT_LAYERS) - 1)

            # gather attention for whatever layers were captured since the last checkpoint
            window_layers = [l for l in attn_capture.keys()]
            stacked = np.stack([attn_capture[l][text_start:, deferred_local].mean(axis=0) for l in window_layers])
            mean_vals = stacked.mean(axis=0)
            var_vals = stacked.var(axis=0)
            attn_capture.clear()  # reset capture buffer for the next window

            keep_local, drop_local, defer_local = score_and_gate(
                deferred_local, mean_vals, var_vals, force_resolve=is_final
            )

            seq_len_now = hidden_states.shape[1]
            drop_set = set(drop_local.tolist())
            keep_mask = torch.ones(seq_len_now, dtype=torch.bool, device=hidden_states.device)
            for d in drop_set:
                keep_mask[d] = False
            survivor_idx = torch.nonzero(keep_mask).squeeze(-1)

            print(f"  [checkpoint @ layer {i}] keep={len(keep_local)} drop={len(drop_local)} "
                  f"defer={len(defer_local)}  (pool was {len(deferred_local)})")

            hidden_states, position_embeddings, causal_mask_mapping = apply_checkpoint_stub(
                hidden_states, position_embeddings, causal_mask_mapping, survivor_idx
            )

            # remap remaining indices into the NEW (post-drop) coordinate space
            surv_list = survivor_idx.cpu().numpy()
            remap = {old: new for new, old in enumerate(surv_list)}
            deferred_local = np.array([remap[d] for d in defer_local if d in remap], dtype=int)
            text_start = remap.get(text_start, text_start - len(drop_set))

    hidden_states = self.norm(hidden_states)
    return BaseModelOutputWithPast(last_hidden_state=hidden_states, past_key_values=past_key_values)

In [18]:
print("model:", "model" in dir())
print("processor:", "processor" in dir())
print("SAMPLES:", "SAMPLES" in dir())
print("dap_forward_v3:", "dap_forward_v3" in dir())
print("score_and_gate:", "score_and_gate" in dir())

model: True
processor: True
SAMPLES: True
dap_forward_v3: True
score_and_gate: True


In [19]:
# Cell D7 — coherence test: checkpoints 1+2+3 with real deferral, 5 samples
def run_dap_v3_test(sample, use_dap):
    image = Image.open(os.path.join(IMG_DIR, sample["image_file"])).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": f"{sample['question']} Answer the question using a single word or phrase."}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda:0")

    ids = inputs.input_ids[0]
    vis_mask = (ids == IMG_TOKEN_ID)
    vis_idx = torch.nonzero(vis_mask).squeeze(-1)
    vis_start, vis_end = vis_idx[0].item(), vis_idx[-1].item() + 1
    text_start = vis_end

    with torch.no_grad():
        inputs_embeds = model.get_input_embeddings()(inputs.input_ids)
        image_embeds = model.get_image_features(inputs.pixel_values, inputs.image_grid_thw).pooler_output
        image_embeds = torch.cat(image_embeds, dim=0).to(inputs_embeds.device, inputs_embeds.dtype)
        mask, _ = model.model.get_placeholder_mask(inputs.input_ids, inputs_embeds=inputs_embeds, image_features=image_embeds)
        inputs_embeds = inputs_embeds.masked_scatter(mask, image_embeds)

        hooks = []
        if use_dap:
            attn_capture.clear()
            hooks = register_early_hooks(text_model, list(range(2, 15)))  # cover all 3 checkpoint windows
            text_model._dap_vis_range = (vis_start, vis_end, text_start)
            original_forward = text_model.forward
            text_model.forward = types.MethodType(dap_forward_v3, text_model)

        current_embeds = inputs_embeds
        generated = None
        for step in range(31):
            out = text_model(inputs_embeds=current_embeds, use_cache=False)
            next_logits = model.lm_head(out.last_hidden_state[:, -1, :])
            next_tok = next_logits.argmax(-1)
            if next_tok.item() == processor.tokenizer.eos_token_id:
                break
            next_embed = model.get_input_embeddings()(next_tok.unsqueeze(0))
            current_embeds = torch.cat([current_embeds, next_embed], dim=1)
            generated = next_tok if generated is None else torch.cat([generated, next_tok])

        if use_dap:
            text_model.forward = original_forward
            unregister_hooks(hooks)

    if generated is None:
        return ""
    return processor.tokenizer.decode(generated, skip_special_tokens=True).strip()


print("="*60)
for s in SAMPLES[:5]:
    baseline = run_dap_v3_test(s, use_dap=False)
    pruned = run_dap_v3_test(s, use_dap=True)
    match = "SAME" if baseline.lower() == pruned.lower() else "DIFFERENT"
    print(f"Q: {s['question']}")
    print(f"  baseline: {baseline!r}")
    print(f"  pruned  : {pruned!r}  [{match}]")
    print("-"*60)

  [checkpoint @ layer 8] keep=80 drop=75 defer=111  (pool was 266)
  [checkpoint @ layer 11] keep=34 drop=31 defer=46  (pool was 111)
  [checkpoint @ layer 14] keep=30 drop=16 defer=0  (pool was 46)
  [checkpoint @ layer 8] keep=80 drop=75 defer=111  (pool was 266)
  [checkpoint @ layer 11] keep=34 drop=31 defer=46  (pool was 111)
  [checkpoint @ layer 14] keep=30 drop=16 defer=0  (pool was 46)
Q: Did one fall on the ground?
  baseline: 'no'
  pruned  : 'no'  [SAME]
------------------------------------------------------------
  [checkpoint @ layer 8] keep=104 drop=97 defer=144  (pool was 345)
  [checkpoint @ layer 11] keep=43 drop=41 defer=60  (pool was 144)
  [checkpoint @ layer 14] keep=39 drop=21 defer=0  (pool was 60)
  [checkpoint @ layer 8] keep=104 drop=97 defer=144  (pool was 345)
  [checkpoint @ layer 11] keep=43 drop=41 defer=60  (pool was 144)
  [checkpoint @ layer 14] keep=39 drop=21 defer=0  (pool was 60)
Q: How many birds are in the shade?
  baseline: '0'
  pruned  : '0' 

In [20]:
# Cell D8 — larger accuracy check: full 3-checkpoint DAP vs. baseline, N=40
import re

def vqa_accuracy(predicted, ground_truth_answers):
    def normalise(s):
        s = s.lower().strip()
        s = re.sub(r"[^\w\s]", "", s)
        for a in ["a ", "an ", "the "]:
            if s.startswith(a): s = s[len(a):]
        return s.strip()
    p = normalise(predicted)
    m = sum(1 for a in ground_truth_answers if normalise(a) == p)
    return min(m / 3, 1.0)


N_CHECK = 40
results = []

for idx, s in enumerate(SAMPLES[:N_CHECK]):
    baseline_pred = run_dap_v3_test(s, use_dap=False)
    pruned_pred = run_dap_v3_test(s, use_dap=True)

    baseline_acc = vqa_accuracy(baseline_pred, s["answers"])
    pruned_acc = vqa_accuracy(pruned_pred, s["answers"])

    results.append({
        "idx": idx,
        "question": s["question"],
        "baseline_pred": baseline_pred,
        "pruned_pred": pruned_pred,
        "baseline_acc": baseline_acc,
        "pruned_acc": pruned_acc,
        "match": baseline_pred.lower() == pruned_pred.lower(),
    })

    if (idx + 1) % 5 == 0:
        print(f"  ...{idx+1}/{N_CHECK} done")

print("\n" + "="*60)
print("SUMMARY — Full 3-checkpoint DAP (KEEP_PCTILE=70, DROP_PCTILE=40)")
print("="*60)

import statistics as st
baseline_mean = st.mean(r["baseline_acc"] for r in results)
pruned_mean = st.mean(r["pruned_acc"] for r in results)
exact_match_rate = sum(r["match"] for r in results) / len(results)

print(f"N = {len(results)}")
print(f"Baseline accuracy: {baseline_mean:.4f}")
print(f"Pruned accuracy:   {pruned_mean:.4f}")
print(f"Accuracy delta:    {pruned_mean - baseline_mean:+.4f}")
print(f"Exact prediction match rate: {exact_match_rate:.2%}")

print("\n--- Cases where predictions DIFFERED ---")
diffs = [r for r in results if not r["match"]]
if diffs:
    for r in diffs:
        print(f"  Q: {r['question']}")
        print(f"    baseline: {r['baseline_pred']!r} (acc={r['baseline_acc']:.2f})")
        print(f"    pruned  : {r['pruned_pred']!r} (acc={r['pruned_acc']:.2f})")
else:
    print("  None - all predictions matched exactly.")

  [checkpoint @ layer 8] keep=80 drop=75 defer=111  (pool was 266)
  [checkpoint @ layer 11] keep=34 drop=31 defer=46  (pool was 111)
  [checkpoint @ layer 14] keep=30 drop=16 defer=0  (pool was 46)
  [checkpoint @ layer 8] keep=80 drop=75 defer=111  (pool was 266)
  [checkpoint @ layer 11] keep=34 drop=31 defer=46  (pool was 111)
  [checkpoint @ layer 14] keep=30 drop=16 defer=0  (pool was 46)
  [checkpoint @ layer 8] keep=104 drop=97 defer=144  (pool was 345)
  [checkpoint @ layer 11] keep=43 drop=41 defer=60  (pool was 144)
  [checkpoint @ layer 14] keep=39 drop=21 defer=0  (pool was 60)
  [checkpoint @ layer 8] keep=104 drop=97 defer=144  (pool was 345)
  [checkpoint @ layer 11] keep=43 drop=41 defer=60  (pool was 144)
  [checkpoint @ layer 14] keep=39 drop=21 defer=0  (pool was 60)
  [checkpoint @ layer 8] keep=159 drop=148 defer=222  (pool was 529)
  [checkpoint @ layer 11] keep=67 drop=62 defer=93  (pool was 222)
  [checkpoint @ layer 14] keep=60 drop=33 defer=0  (pool was 93)
 

In [23]:
# Cell D9 — Zeus init + idle baseline (this exploratory notebook never had this)
from zeus.monitor import ZeusMonitor
import time

monitor = ZeusMonitor(gpu_indices=[0])
print("Zeus initialized.")

print("Measuring idle baseline (30 seconds)...")
idle_readings = []
for _ in range(300):
    idle_readings.append(monitor.gpus.getInstantPowerUsage(0) / 1000)
    time.sleep(0.1)

IDLE_POWER_W = sum(idle_readings) / len(idle_readings)
print(f"Idle baseline: {IDLE_POWER_W:.3f} W")

/opt/rocm/lib/libamd_smi.so: cannot open shared object file: No such file or directory
Unable to find libamd_smi.so library try installing amd-smi-lib from your package manager
[2026-07-27 20:37:58,688] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-07-27 20:37:58,690] [zeus.device.cpu.rapl](rapl.py:137) RAPL is not supported on this CPU.
[2026-07-27 20:37:58,690] [zeus.monitor.energy](energy.py:209) Monitoring GPU indices [0].
[2026-07-27 20:37:58,691] [zeus.monitor.energy](energy.py:210) Monitoring CPU indices []
Zeus initialized.
Measuring idle baseline (30 seconds)...
Idle baseline: 33.240 W


In [24]:
# Cell D10 — energy comparison, baseline vs. DAP, per-sample energy windows
def run_dap_energy_test(sample, use_dap):
    image = Image.open(os.path.join(IMG_DIR, sample["image_file"])).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": f"{sample['question']} Answer the question using a single word or phrase."}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda:0")

    ids = inputs.input_ids[0]
    vis_mask = (ids == IMG_TOKEN_ID)
    vis_idx = torch.nonzero(vis_mask).squeeze(-1)
    vis_start, vis_end = vis_idx[0].item(), vis_idx[-1].item() + 1
    text_start = vis_end

    with torch.no_grad():
        inputs_embeds = model.get_input_embeddings()(inputs.input_ids)
        image_embeds = model.get_image_features(inputs.pixel_values, inputs.image_grid_thw).pooler_output
        image_embeds = torch.cat(image_embeds, dim=0).to(inputs_embeds.device, inputs_embeds.dtype)
        mask, _ = model.model.get_placeholder_mask(inputs.input_ids, inputs_embeds=inputs_embeds, image_features=image_embeds)
        inputs_embeds = inputs_embeds.masked_scatter(mask, image_embeds)

        hooks = []
        if use_dap:
            attn_capture.clear()
            hooks = register_early_hooks(text_model, list(range(2, 15)))
            text_model._dap_vis_range = (vis_start, vis_end, text_start)
            original_forward = text_model.forward
            text_model.forward = types.MethodType(dap_forward_v3, text_model)

        # --- energy window wraps the WHOLE generation, one window per sample ---
        window_name = f"{'dap' if use_dap else 'base'}_{sample['sample_idx']}"
        monitor.begin_window(window_name)
        t0 = time.perf_counter()

        current_embeds = inputs_embeds
        generated = None
        for step in range(31):
            out = text_model(inputs_embeds=current_embeds, use_cache=False)
            next_logits = model.lm_head(out.last_hidden_state[:, -1, :])
            next_tok = next_logits.argmax(-1)
            if next_tok.item() == processor.tokenizer.eos_token_id:
                break
            next_embed = model.get_input_embeddings()(next_tok.unsqueeze(0))
            current_embeds = torch.cat([current_embeds, next_embed], dim=1)
            generated = next_tok if generated is None else torch.cat([generated, next_tok])

        elapsed = time.perf_counter() - t0
        measurement = monitor.end_window(window_name)
        # ---------------------------------------------------------------------

        if use_dap:
            text_model.forward = original_forward
            unregister_hooks(hooks)

    tokens_generated = 0 if generated is None else generated.shape[0]
    predicted = "" if generated is None else processor.tokenizer.decode(generated, skip_special_tokens=True).strip()

    energy_gross = measurement.total_energy
    avg_power = energy_gross / elapsed if elapsed > 0 else 0
    energy_net = max((avg_power - IDLE_POWER_W) * elapsed, 0)

    return {
        "predicted": predicted,
        "tokens_generated": tokens_generated,
        "elapsed_s": elapsed,
        "energy_gross_j": energy_gross,
        "energy_net_j": energy_net,
        "avg_power_w": avg_power,
    }


N_ENERGY = 25
energy_results = []

for idx, s in enumerate(SAMPLES[:N_ENERGY]):
    base = run_dap_energy_test(s, use_dap=False)
    dap = run_dap_energy_test(s, use_dap=True)

    energy_results.append({
        "idx": idx,
        "question": s["question"],
        "base_acc": vqa_accuracy(base["predicted"], s["answers"]),
        "dap_acc": vqa_accuracy(dap["predicted"], s["answers"]),
        "base_energy": base["energy_net_j"],
        "dap_energy": dap["energy_net_j"],
        "base_elapsed": base["elapsed_s"],
        "dap_elapsed": dap["elapsed_s"],
        "base_tokens": base["tokens_generated"],
        "dap_tokens": dap["tokens_generated"],
    })

    if (idx + 1) % 5 == 0:
        print(f"  ...{idx+1}/{N_ENERGY} done")

print("\n" + "="*60)
print("ENERGY COMPARISON — Baseline vs Full 3-Checkpoint DAP")
print("="*60)

import statistics as st
base_e = st.mean(r["base_energy"] for r in energy_results)
dap_e = st.mean(r["dap_energy"] for r in energy_results)
base_t = st.mean(r["base_elapsed"] for r in energy_results)
dap_t = st.mean(r["dap_elapsed"] for r in energy_results)
base_acc = st.mean(r["base_acc"] for r in energy_results)
dap_acc = st.mean(r["dap_acc"] for r in energy_results)

print(f"N = {N_ENERGY}")
print(f"Baseline energy_net (J): {base_e:.2f}")
print(f"DAP energy_net (J):      {dap_e:.2f}")
print(f"Energy reduction:        {100*(1 - dap_e/base_e):.1f}%")
print()
print(f"Baseline elapsed (s):    {base_t:.2f}")
print(f"DAP elapsed (s):         {dap_t:.2f}")
print(f"Latency reduction:       {100*(1 - dap_t/base_t):.1f}%")
print()
print(f"Baseline accuracy:       {base_acc:.4f}")
print(f"DAP accuracy:            {dap_acc:.4f}")
print(f"Accuracy delta:          {dap_acc - base_acc:+.4f}")

[2026-07-27 20:38:54,048] [zeus.utils.framework](framework.py:25) PyTorch with CUDA support is available.
  [checkpoint @ layer 8] keep=80 drop=75 defer=111  (pool was 266)
  [checkpoint @ layer 11] keep=34 drop=31 defer=46  (pool was 111)
  [checkpoint @ layer 14] keep=30 drop=16 defer=0  (pool was 46)
  [checkpoint @ layer 8] keep=80 drop=75 defer=111  (pool was 266)
  [checkpoint @ layer 11] keep=34 drop=31 defer=46  (pool was 111)
  [checkpoint @ layer 14] keep=30 drop=16 defer=0  (pool was 46)
  [checkpoint @ layer 8] keep=104 drop=97 defer=144  (pool was 345)
  [checkpoint @ layer 11] keep=43 drop=41 defer=60  (pool was 144)
  [checkpoint @ layer 14] keep=39 drop=21 defer=0  (pool was 60)
  [checkpoint @ layer 8] keep=104 drop=97 defer=144  (pool was 345)
  [checkpoint @ layer 11] keep=43 drop=41 defer=60  (pool was 144)
  [checkpoint @ layer 14] keep=39 drop=21 defer=0  (pool was 60)
  [checkpoint @ layer 8] keep=159 drop=148 defer=222  (pool was 529)
  [checkpoint @ layer 11] k

In [25]:
# Cell E1 — KV-cache fidelity test, 0% pruning, manual past_key_values management
def generate_with_manual_cache(inputs_embeds, max_new_tokens=31):
    past_key_values = None
    current_embeds = inputs_embeds
    generated = None

    for step in range(max_new_tokens):
        with torch.no_grad():
            out = text_model(
                inputs_embeds=current_embeds if past_key_values is None else current_embeds[:, -1:, :],
                past_key_values=past_key_values,
                use_cache=True,
            )
        past_key_values = out.past_key_values
        next_logits = model.lm_head(out.last_hidden_state[:, -1, :])
        next_tok = next_logits.argmax(-1)
        if next_tok.item() == processor.tokenizer.eos_token_id:
            break
        next_embed = model.get_input_embeddings()(next_tok.unsqueeze(0))
        current_embeds = torch.cat([current_embeds, next_embed], dim=1)
        generated = next_tok if generated is None else torch.cat([generated, next_tok])

    return generated

In [28]:
# Cell E2 — compare cached generation vs. no-cache baseline, 0% pruning
print("="*60)
for s in SAMPLES[:5]:
    image = Image.open(os.path.join(IMG_DIR, s["image_file"])).convert("RGB")
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": f"{s['question']} Answer the question using a single word or phrase."}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda:0")

    with torch.no_grad():
        inputs_embeds = model.get_input_embeddings()(inputs.input_ids)
        image_embeds = model.get_image_features(inputs.pixel_values, inputs.image_grid_thw).pooler_output
        image_embeds = torch.cat(image_embeds, dim=0).to(inputs_embeds.device, inputs_embeds.dtype)
        mask, _ = model.model.get_placeholder_mask(inputs.input_ids, inputs_embeds=inputs_embeds, image_features=image_embeds)
        inputs_embeds = inputs_embeds.masked_scatter(mask, image_embeds)

        generated_cached = generate_with_manual_cache(inputs_embeds)
        generated_nocache = generate_no_cache(inputs_embeds)

    cached_pred = "" if generated_cached is None else processor.tokenizer.decode(generated_cached, skip_special_tokens=True).strip()
    nocache_pred = "" if generated_nocache is None else processor.tokenizer.decode(generated_nocache, skip_special_tokens=True).strip()

    match = "SAME" if nocache_pred.lower() == cached_pred.lower() else "DIFFERENT"
    print(f"Q: {s['question']}")
    print(f"  no-cache: {nocache_pred!r}")
    print(f"  cached  : {cached_pred!r}  [{match}]")
    print("-"*60)

Q: Did one fall on the ground?
  no-cache: 'no'
  cached  : 'no'  [SAME]
------------------------------------------------------------
Q: How many birds are in the shade?
  no-cache: '0'
  cached  : '0'  [SAME]
------------------------------------------------------------
Q: What is on his hands?
  no-cache: 'gloves'
  cached  : 'gloves'  [SAME]
------------------------------------------------------------
Q: What is the animal sitting on?
  no-cache: 'backpack'
  cached  : 'backpack'  [SAME]
------------------------------------------------------------
Q: Where are the arrows pointing on the yellow sign?
  no-cache: 'both ways'
  cached  : 'both ways'  [SAME]
------------------------------------------------------------


In [27]:
# Cell E2b — self-contained no-cache baseline (replaces dependency on old run_dap_test)
def generate_no_cache(inputs_embeds, max_new_tokens=31):
    current_embeds = inputs_embeds
    generated = None
    for step in range(max_new_tokens):
        with torch.no_grad():
            out = text_model(inputs_embeds=current_embeds, use_cache=False)
        next_logits = model.lm_head(out.last_hidden_state[:, -1, :])
        next_tok = next_logits.argmax(-1)
        if next_tok.item() == processor.tokenizer.eos_token_id:
            break
        next_embed = model.get_input_embeddings()(next_tok.unsqueeze(0))
        current_embeds = torch.cat([current_embeds, next_embed], dim=1)
        generated = next_tok if generated is None else torch.cat([generated, next_tok])
    return generated

In [29]:
# Cell E3 — inspect the actual structure of past_key_values before writing slicing code
s = SAMPLES[0]
image = Image.open(os.path.join(IMG_DIR, s["image_file"])).convert("RGB")
messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": f"{s['question']} Answer the question using a single word or phrase."}]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda:0")

with torch.no_grad():
    inputs_embeds = model.get_input_embeddings()(inputs.input_ids)
    image_embeds = model.get_image_features(inputs.pixel_values, inputs.image_grid_thw).pooler_output
    image_embeds = torch.cat(image_embeds, dim=0).to(inputs_embeds.device, inputs_embeds.dtype)
    mask, _ = model.model.get_placeholder_mask(inputs.input_ids, inputs_embeds=inputs_embeds, image_features=image_embeds)
    inputs_embeds = inputs_embeds.masked_scatter(mask, image_embeds)

    out = text_model(inputs_embeds=inputs_embeds, use_cache=True)

pkv = out.past_key_values
print("Type of past_key_values:", type(pkv))
print("Has 'layers' attr:", hasattr(pkv, "layers"))
print("Has 'key_cache' attr:", hasattr(pkv, "key_cache"))
print("Is it indexable/iterable as a list?", hasattr(pkv, "__len__"))
if hasattr(pkv, "__len__"):
    print("Length:", len(pkv))
    first = pkv[0] if hasattr(pkv, "__getitem__") else None
    print("First element type:", type(first))
    if first is not None:
        print("First element (key, value) shapes:", first[0].shape, first[1].shape)

Type of past_key_values: <class 'transformers.cache_utils.DynamicCache'>
Has 'layers' attr: True
Has 'key_cache' attr: False
Is it indexable/iterable as a list? True
Length: 28
First element type: <class 'NoneType'>


In [30]:
# Cell E4 — inspect the actual per-layer cache structure via .layers
layer0_cache = pkv.layers[0]
print("Type of pkv.layers[0]:", type(layer0_cache))
print("Attributes:", [a for a in dir(layer0_cache) if not a.startswith('_')])

Type of pkv.layers[0]: <class 'transformers.cache_utils.DynamicLayer'>
Attributes: ['batch_repeat_interleave', 'batch_select_indices', 'crop', 'device', 'dtype', 'get_mask_sizes', 'get_max_cache_shape', 'get_max_length', 'get_seq_length', 'is_compileable', 'is_initialized', 'is_sliding', 'keys', 'lazy_initialization', 'offload', 'prefetch', 'reorder_cache', 'reset', 'supports_early_init', 'update', 'values']


In [31]:
# Cell E5 — confirm shapes and inspect the built-in slicing methods
print("Keys shape:", layer0_cache.keys.shape)
print("Values shape:", layer0_cache.values.shape)
print()

import inspect
print("--- batch_select_indices ---")
print(inspect.getsource(layer0_cache.batch_select_indices))

Keys shape: torch.Size([1, 4, 304, 128])
Values shape: torch.Size([1, 4, 304, 128])

--- batch_select_indices ---
    def batch_select_indices(self, indices: torch.Tensor) -> None:
        """Only keep the `indices` in the batch dimension of the cache."""
        if self.get_seq_length() > 0:
            self.keys = self.keys[indices, ...]
            self.values = self.values[indices, ...]



In [32]:
# Cell E6 — checkpoint-aware cache slicing, 0% pruning (extends apply_checkpoint_stub to also slice past_key_values)

def apply_checkpoint_stub_with_cache(hidden_states, position_embeddings, causal_mask_mapping,
                                       past_key_values, survivor_idx, num_layers_so_far):
    hidden_states = hidden_states[:, survivor_idx, :]
    cos, sin = position_embeddings
    cos = cos[..., survivor_idx, :]
    sin = sin[..., survivor_idx, :]
    position_embeddings = (cos, sin)
    new_mask_mapping = {k: m[:, :, survivor_idx, :][:, :, :, survivor_idx] for k, m in causal_mask_mapping.items()}

    # slice every layer's cache that has been populated so far (layers 0..num_layers_so_far-1)
    if past_key_values is not None:
        for layer_idx in range(num_layers_so_far):
            layer_cache = past_key_values.layers[layer_idx]
            if layer_cache.get_seq_length() > 0:
                layer_cache.keys = layer_cache.keys[:, :, survivor_idx, :]
                layer_cache.values = layer_cache.values[:, :, survivor_idx, :]

    return hidden_states, position_embeddings, new_mask_mapping, past_key_values


# Quick single-sample test: 0% pruning, but now routing through the cache-slicing path
s = SAMPLES[0]
image = Image.open(os.path.join(IMG_DIR, s["image_file"])).convert("RGB")
messages = [{"role": "user", "content": [
    {"type": "image"},
    {"type": "text", "text": f"{s['question']} Answer the question using a single word or phrase."}]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt").to("cuda:0")

with torch.no_grad():
    inputs_embeds = model.get_input_embeddings()(inputs.input_ids)
    image_embeds = model.get_image_features(inputs.pixel_values, inputs.image_grid_thw).pooler_output
    image_embeds = torch.cat(image_embeds, dim=0).to(inputs_embeds.device, inputs_embeds.dtype)
    mask, _ = model.model.get_placeholder_mask(inputs.input_ids, inputs_embeds=inputs_embeds, image_features=image_embeds)
    inputs_embeds = inputs_embeds.masked_scatter(mask, image_embeds)

    out = text_model(inputs_embeds=inputs_embeds, use_cache=True)
    seq_len = inputs_embeds.shape[1]
    survivor_idx = torch.arange(seq_len, device=inputs_embeds.device)  # 0% pruning = identity

    pkv = out.past_key_values
    # test slicing at "checkpoint 1" (pretend layer 8 boundary) with identity indices
    for layer_idx in range(9):  # layers 0-8 populated so far
        lc = pkv.layers[layer_idx]
        before_shape = lc.keys.shape
        lc.keys = lc.keys[:, :, survivor_idx, :]
        lc.values = lc.values[:, :, survivor_idx, :]
        after_shape = lc.keys.shape
    print(f"Layer 8 cache shape before: {before_shape}, after (identity slice): {after_shape}")
    print("Shapes match (expected for 0% pruning):", before_shape == after_shape)

Layer 8 cache shape before: torch.Size([1, 4, 304, 128]), after (identity slice): torch.Size([1, 4, 304, 128])
Shapes match (expected for 0% pruning): True
